# WDL — Module 3 · Session 13 — Extremes and intensity
### Reading notebook

| | |
|---|---|
| **Course** | Hydroinformatics in a Changing Climate: Water Data Literacy for All |
| **Department** | Civil Engineering · CHRIST (Deemed to be University) |
| **Instructor** | Dr Arpan Pradhan |
| **Module** | Module 3 — Reading Rainfall |
| **Session** | Session 13 of 30 |
| **CO** | CO2, CO3 |
| **Topics** | IMD daily rainfall intensity categories; wettest days; how much of the year falls in a few days |
| **Water hook** | *Heavy, very heavy, extremely heavy* |

---
> **How to use this notebook.** No coding needed. Read each explanation, then press ▶ on the box below it
> (or click the box and press `Shift + Enter`). Some boxes have sliders, number boxes or dropdowns: change them, then press ▶ again.
> Go to **File → Save a copy in Drive** first, so the notebook is yours.

## Why this session matters
"Heavy rain warning", "very heavy", "extremely heavy": IMD uses exact thresholds for these words. This session shows how
often each kind of rainy day happens, and how much of the year's water comes from a few heavy days.

In [ ]:
#@title ▶ Step 0 · Get ready (press ▶ once before anything else)
import os, json, warnings
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
_orig_bar = plt.Axes.bar
def _bar_int_years(self, x, *a, **k):        # whole-number year ticks on bar charts of years
    out = _orig_bar(self, x, *a, **k)
    try:
        xs = np.asarray(x)
        if xs.dtype.kind in "iu" and xs.min() > 1800: self.xaxis.set_major_locator(MaxNLocator(integer=True))
    except Exception: pass
    return out
plt.Axes.bar = _bar_int_years
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110, "font.size": 11,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, RED, GREY, TEAL, GOLD = "#2C6E9B", "#DD8452", "#C44E52", "#8C8C8C", "#1F7A8C", "#C9962E"
AUTO = os.environ.get("WDL_SELFTEST") == "1"

def ok(msg="Correct!"):  display(Markdown(f"✅ **{msg}**"))
def no(msg="Not quite. Check your working and try again."): display(Markdown(f"❌ {msg}"))
def _num(x):
    try: return float(str(x).replace(",", "").replace("%", "").strip())
    except (TypeError, ValueError): return None
def close(a, b, tol=0.02):
    return a is not None and abs(a - b) <= max(abs(b) * tol, 0.01)
def check_num(ans, exp, tol=0.02, fmt="{:,.1f}", hint=""):
    if AUTO: ans = exp
    v = _num(ans)
    if v is None: no("Type your answer in the box first (numbers only)."); return False
    if close(v, float(exp), tol): ok("Correct: " + fmt.format(exp)); return True
    no("Not quite. " + hint); return False
def check_choice(ans, exp, explain="", hint=""):
    if AUTO: ans = exp
    if ans in ("", "choose"): no("Choose an answer from the dropdown first."); return False
    if ans == exp: ok(("Correct. " + explain).strip()); return True
    no(("Not quite. " + hint).strip()); return False

def make_sample_district():
    """Illustrative, synthetic daily rainfall for a South Indian district, 1981-2024."""
    rng = np.random.default_rng(1981)
    days = pd.date_range("1981-01-01", "2024-12-31", freq="D")
    p_wet = np.array([0.03, 0.03, 0.06, 0.15, 0.30, 0.35, 0.42, 0.42, 0.45, 0.45, 0.28, 0.08])
    mu = np.array([4, 4, 7, 11, 13, 8, 8, 9, 13, 14, 12, 6], float)
    years = np.arange(1981, 2025)
    yf = dict(zip(years, np.exp(rng.normal(0, 0.17, len(years)))))
    m = days.month.values - 1
    t = days.year.values - 1981
    yfac = np.array([yf[y] for y in days.year.values])
    p = np.clip(p_wet[m] * (1 - 0.0025 * t) * yfac ** 0.5, 0, 0.95)
    inten = mu[m] * (1 + 0.005 * t) * yfac ** 0.5
    wet = rng.random(len(days)) < p
    amt = rng.gamma(0.7, inten / 0.7)
    rain = np.round(np.where(wet, amt, 0.0), 1)
    for d, v in [("1991-11-14", 142.0), ("2005-10-24", 186.2), ("2017-08-14", 151.4), ("2022-09-05", 168.9)]:
        rain[days.get_loc(pd.Timestamp(d))] = v
    return pd.Series(rain, index=days, name="rain_mm")

def make_messy_record(sample):
    """Three years of the sample record with typical gauge-record problems added."""
    df = sample["2018":"2020"].rename("rain_mm").reset_index()
    df.columns = ["date", "rain_mm"]
    flag_dates = ["2018-02-03", "2018-05-19", "2018-08-07", "2018-11-25", "2019-01-14",
                  "2019-04-02", "2019-09-18", "2019-12-30", "2020-06-06", "2020-10-11"]
    blank_dates = ["2018-07-21", "2019-03-09", "2020-02-17", "2020-08-28"]
    df.loc[df.date.isin(pd.to_datetime(flag_dates)), "rain_mm"] = -999.0
    df.loc[df.date.isin(pd.to_datetime(blank_dates)), "rain_mm"] = np.nan
    df.loc[df.date == "2018-09-12", "rain_mm"] = 999.9      # typing error
    df.loc[df.date == "2020-04-23", "rain_mm"] = -3.0       # impossible negative
    df = df[~df.date.between("2019-07-01", "2019-07-12")]  # 12 days never recorded
    dup = df[df.date.isin(pd.to_datetime(["2018-06-15", "2019-10-03", "2020-11-20"]))]
    df = pd.concat([df, dup]).sort_values("date", kind="stable").reset_index(drop=True)
    df["date"] = df.date.dt.strftime("%Y-%m-%d")
    return df

SAMPLE_URL = "https://raw.githubusercontent.com/PradhanArpan/waterliteracy/main/data/sample_district_daily_1981_2024.csv"
try:
    sample = pd.read_csv(SAMPLE_URL, parse_dates=["date"], index_col="date")["rain_mm"]
except Exception:
    sample = make_sample_district()   # works offline too: same numbers

def find_place(name):
    """Place name -> (lat, lon, label) using Open-Meteo geocoding. Prefers places in India."""
    parts = [p.strip() for p in str(name).split(",") if p.strip()]
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                     params={"name": parts[0], "count": 10, "language": "en", "format": "json"}, timeout=20)
    res = r.json().get("results", []) or []
    res = [x for x in res if x.get("country_code") == "IN"] or res
    if len(parts) > 1:
        res = [x for x in res if parts[1].lower() in str(x.get("admin1", "")).lower()] or res
    if not res: raise ValueError(f"No place found for '{name}'")
    x = res[0]
    return x["latitude"], x["longitude"], f'{x["name"]}, {x.get("admin1", "")}'.strip(", ")

def fetch_power(lat, lon, start=1981, end=2024):
    """Daily rainfall (mm/day) from NASA POWER (PRECTOTCORR) for one point."""
    r = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", timeout=90, params={
        "parameters": "PRECTOTCORR", "community": "AG", "latitude": round(lat, 4), "longitude": round(lon, 4),
        "start": f"{start}0101", "end": f"{end}1231", "format": "JSON"})
    r.raise_for_status()
    d = r.json()["properties"]["parameter"]["PRECTOTCORR"]
    s = pd.Series(d, dtype=float)
    s.index = pd.to_datetime(s.index, format="%Y%m%d")
    s[s < -98] = np.nan
    return s.round(1).rename("rain_mm")

def fetch_era5(lat, lon, start="2021-01-01", end="2021-12-31"):
    """Daily rainfall (mm/day) from the ERA5 reanalysis via Open-Meteo."""
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=60, params={
        "latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
        "daily": "precipitation_sum", "timezone": "Asia/Kolkata"})
    r.raise_for_status()
    d = r.json()["daily"]
    return pd.Series(d["precipitation_sum"], index=pd.to_datetime(d["time"]), dtype=float).rename("rain_mm")

def load_rain(source, place="", lat=0.0, lon=0.0):
    """Returns (daily rainfall Series, label, lat, lon). Falls back to the sample district."""
    if source.startswith("Sample") or AUTO:
        return sample, "Sample district (synthetic)", None, None
    try:
        if not (lat and lon):
            lat, lon, place = find_place(place)
        cache = f"rain_{lat:.3f}_{lon:.3f}.csv"
        if os.path.exists(cache):
            s = pd.read_csv(cache, parse_dates=["date"], index_col="date")["rain_mm"]
        else:
            s = fetch_power(lat, lon)
            s.rename_axis("date").to_csv(cache)
        return s, f"{place} ({lat:.2f}°N, {lon:.2f}°E) · NASA POWER", lat, lon
    except Exception as e:
        display(Markdown(f"⚠️ Could not download data for **{place or 'your place'}** ({type(e).__name__}). "
                         "Using the sample district instead. Check the place name or type latitude and longitude."))
        return sample, "Sample district (synthetic)", None, None

def annual(s):  return s.groupby(s.index.year).sum(min_count=300)
def monsoon(s):
    m = s[s.index.month.isin([6, 7, 8, 9])]
    return m.groupby(m.index.year).sum(min_count=100)
display(Markdown("✅ **Ready.** Tools loaded. Now run **Step 1** to choose your data."))

---
## Step 1 · Choose your data
> **About the data.** *Sample district* is an illustrative, synthetic 44-year record (1981–2024) made for practice;
> it is not an official record, and answers in the practice tasks and the Sheet use it.
> *My place* downloads real daily rainfall for any location from **NASA POWER** (satellite and model based, from 1981).
> Type a town or district name, or enter latitude and longitude.

In [ ]:
#@title ▶ Step 1 · Choose your data { run: "auto" }
data_source = "Sample district (works offline)" #@param ["Sample district (works offline)", "My place (live download from NASA POWER)"]
place = "Mysuru, Karnataka" #@param {type:"string"}
latitude = 0 #@param {type:"number"}
longitude = 0 #@param {type:"number"}
rain, label, LAT, LON = load_rain(data_source, place, latitude, longitude)
display(Markdown(f"**Data:** {label} · {rain.index.min():%d %b %Y} to {rain.index.max():%d %b %Y} · "
                 f"{len(rain):,} days"))

In [ ]:
#@title ▶ Step 2 · Prepare the tables (press ▶)
A = annual(rain)                                    # annual totals
SW = monsoon(rain)                                  # June–September totals
NORMAL = A.loc[1991:2020].mean()                    # 30-year normal, WMO standard period
NORMAL_SW = SW.loc[1991:2020].mean()
def imd_category(dep):
    if dep <= -99.5: return "No rain"
    if dep <= -60: return "Large deficient"
    if dep <= -20: return "Deficient"
    if dep < 20: return "Normal"
    if dep < 60: return "Excess"
    return "Large excess"
CAT_COLOURS = {"Large deficient": "#8E2C2C", "Deficient": RED, "Normal": GREY, "Excess": BLUE, "Large excess": "#1B3F66", "No rain": "black"}
INTENSITY = [("Very light", 0.1, 2.4), ("Light", 2.5, 15.5), ("Moderate", 15.6, 64.4), ("Heavy", 64.5, 115.5),
             ("Very heavy", 115.6, 204.4), ("Extremely heavy", 204.5, 10000)]
def intensity_class(x):
    for name, lo, hi in INTENSITY:
        if lo <= x <= hi: return name
    return "No rain"
AMAX = rain.groupby(rain.index.year).max()          # annual maximum daily rainfall
display(Markdown(f"Tables ready. 1991–2020 normal: **{NORMAL:,.0f} mm** per year, **{NORMAL_SW:,.0f} mm** in the June–September monsoon."))

---
## Part 1 · IMD's words for daily rainfall

| Daily rainfall (24 hours) | IMD term |
|---|---|
| 0.1 – 2.4 mm | Very light rain |
| 2.5 – 15.5 mm | Light rain |
| 15.6 – 64.4 mm | Moderate rain |
| 64.5 – 115.5 mm | Heavy rain |
| 115.6 – 204.4 mm | Very heavy rain |
| 204.5 mm or more | Extremely heavy rain |

In [ ]:
#@title ▶ How many days of each kind?
cls = rain[rain > 0].apply(intensity_class)
order = [n for n, _, _ in INTENSITY]
cnt = cls.value_counts().reindex(order).fillna(0).astype(int)
share = rain[rain > 0].groupby(cls).sum().reindex(order).fillna(0) / rain.sum() * 100
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.bar(order, cnt.values, color=BLUE); a1.set_title("Number of days"); a1.tick_params(axis="x", rotation=35)
for i, v in enumerate(cnt.values): a1.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
a2.bar(order, share.values, color=ORANGE); a2.set_title("Share of all rain (%)"); a2.tick_params(axis="x", rotation=35)
for i, v in enumerate(share.values): a2.text(i, v, f"{v:.0f}%", ha="center", va="bottom", fontsize=9)
plt.suptitle(f"{label.split(' · ')[0]}, {rain.index.min().year}–{rain.index.max().year}", fontweight="bold"); plt.tight_layout(); plt.show()

### What to notice
Heavy, very heavy and extremely heavy days are a tiny fraction of all rainy days, yet they bring a large slice of the water.
Those are the days that fill tanks and cause floods.

---
## Part 2 · The wettest days on record
Press ▶.

In [ ]:
#@title ▶ Top days { run: "auto" }
how_many = 10 #@param {type:"slider", min:5, max:25, step:1}
top = rain.sort_values(ascending=False).head(how_many)
t = pd.DataFrame({"Rain (mm)": top.round(1), "IMD term": top.apply(intensity_class), "Month": top.index.strftime("%B")})
t.index = t.index.strftime("%d %b %Y"); display(t)

---
## Part 3 · The wettest day of every year
This **annual maximum** series is what engineers use to design drains and culverts. Press ▶.

In [ ]:
#@title ▶ Annual maximum daily rainfall
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(AMAX.index, AMAX.values, "o-", color=BLUE)
ax.axhline(64.5, color=ORANGE, ls="--"); ax.text(AMAX.index[0], 66, "Heavy (64.5 mm)", color=ORANGE, fontsize=9)
ax.axhline(115.6, color=RED, ls="--"); ax.text(AMAX.index[0], 117, "Very heavy (115.6 mm)", color=RED, fontsize=9)
ax.set_ylabel("Wettest day of the year (mm)"); ax.set_title("Annual maximum daily rainfall")
plt.tight_layout(); plt.show()

### ⏭️ Next
Open the **Practice activity** and count the extremes yourself. Next session, we turn this series into return periods.